In [1]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px
import numpy as np

# Generate a sample dataset
np.random.seed(42)
df = pd.DataFrame({
    "Date": pd.date_range(start="2023-01-01", periods=100),
    "Category": np.random.choice(["A", "B", "C"], size=100),
    "Value": np.random.randint(10, 100, size=100)
})

# Create the Dash app
app = dash.Dash(__name__)

# Layout of the dashboard
app.layout = html.Div([
    html.H1("Sample Dashboard with Dash", style={"textAlign": "center"}),

    # Dropdown to filter by category
    html.Div([
        html.Label("Select Category:"),
        dcc.Dropdown(
            id="category-dropdown",
            options=[{"label": cat, "value": cat} for cat in df["Category"].unique()],
            value="A",  # Default value
            clearable=False
        )
    ], style={"width": "50%", "margin": "auto"}),

    # Graph to display the data
    dcc.Graph(id="line-chart"),

    # Range slider to filter by date
    html.Div([
        html.Label("Select Date Range:"),
        dcc.RangeSlider(
            id="date-slider",
            min=0,
            max=len(df) - 1,
            step=1,
            value=[0, len(df) - 1],
            marks={i: str(date.date()) for i, date in enumerate(df["Date"][::10])}
        )
    ], style={"width": "80%", "margin": "auto"})
])

# Callback to update the graph based on dropdown and slider
@app.callback(
    Output("line-chart", "figure"),
    [Input("category-dropdown", "value"),
     Input("date-slider", "value")]
)
def update_graph(selected_category, date_range):
    filtered_df = df[
        (df["Category"] == selected_category) &
        (df.index >= date_range[0]) &
        (df.index <= date_range[1])
    ]
    fig = px.line(
        filtered_df, x="Date", y="Value", title=f"Category: {selected_category}",
        labels={"Value": "Value", "Date": "Date"}
    )
    return fig

# Run the app
if __name__ == "__main__":
    app.run_server(debug=True)
